# 🔎 | Análise Exploratória de Dados

### 📚 | Importação de bibliotecas e do conjunto de dados

In [433]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.2f' % x) # Remove notação científica dos outputs dos dados

In [434]:
df_pedidos = pd.read_csv("../data/demanda_pedidos_final.csv")
df_pedidos.head()

,id_produto,quantidade,data_pedido
0,1206997678774,2.00,2025-12-31
1,2483458744482,2.00,2025-12-31
2,3115403067986,2.00,2025-12-31
3,4616333531350,2.00,2025-12-31
4,2124623355660,5.00,2025-12-31


In [435]:
df_produtos = pd.read_csv("../data/demanda_produtos_final.csv")
df_produtos.head()

,id_produto,curva,categoria,fora_de_estoque,indice_de_preco,data_coleta
0,6693862302648,B,Categoria 35,0,0.95,2026-03-12
1,6693862302648,B,Categoria 35,0,0.95,2026-03-13
2,6693862302648,B,Categoria 35,0,1.18,2026-03-14
3,6693862302648,B,Categoria 35,0,1.18,2026-03-15
4,6693862302648,B,Categoria 35,0,0.95,2026-03-16


Com o objetivo de adequar o conjunto de dados à especificação original mostrada pela equipe da Praso, nós decidimos modificar a coluna `fora_de_estoque`, que indica a falta do produto no estoque naquela data (1) ou não (0), para uma coluna exatamente oposta à esta, chamada `estoque`, que indica se o produto tinha estoque na data (1) ou não (0).

In [436]:
# Ajustando tabela de produtos pra ficar igual à especificação
df_produtos["fora_de_estoque"] = df_produtos["fora_de_estoque"].replace({0: 1, 1: 0})
df_produtos.rename(columns={"fora_de_estoque": "estoque"}, inplace=True)

### ⏳ | Tratamento de séries temporais

Dado que os dados que estão sendo trabalhados possuem uma temporalidade determinante, é necessário que consigamos extrair alguns aspectos que podem ser interessantes para análises futuras por meio de agregações dos dados por semanas.

#### Dataset de pedidos

In [437]:
print("====== INFORMAÇÕES GERAIS ======")
print(f"Formato: {df_pedidos.shape[0]} linhas e {df_pedidos.shape[1]} colunas")
print(f"Uso de memória: {df_pedidos.memory_usage(deep=True).sum() / 1024**2:.2f} Mb")

====== INFORMAÇÕES GERAIS ======
Formato: 45807 linhas e 3 colunas
Uso de memória: 3.28 Mb


In [438]:
print("====== INFORMAÇÕES DETALHADAS ======")
print(df_pedidos.info())

====== INFORMAÇÕES DETALHADAS ======
<class 'pandas.DataFrame'>
RangeIndex: 45807 entries, 0 to 45806
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   45807 non-null  int64  
 1   quantidade   45807 non-null  float64
 2   data_pedido  45807 non-null  str    
dtypes: float64(1), int64(1), str(1)
memory usage: 1.0 MB
None


Percebemos que `data_pedido` está sendo classificado como de tipo `object`, entretanto, para caracterizar melhor esse tipo de dado, podemos convertê-lo para um tipo especificamente `datetime`.

Além disso, o `id_produto` não devem ser caracterizado como elemento de tipo numérico porque não representa uma quantidade, mas sim um rótulo específico.

Aplicando a transformação:

In [439]:
df_pedidos['data_pedido'] = pd.to_datetime(df_pedidos['data_pedido'])
df_pedidos['id_produto'] = df_pedidos['id_produto'].astype(str)

Verificando novamente os tipos de dados:

In [440]:
print(df_pedidos.info())

<class 'pandas.DataFrame'>
RangeIndex: 45807 entries, 0 to 45806
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_produto   45807 non-null  str           
 1   quantidade   45807 non-null  float64       
 2   data_pedido  45807 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 1.0 MB
None


A análise e modelagem serão executadas conforme a demanda semanal, mas a forma que o conjunto de dados se apresenta no momento mostra os pedidos estão em granularidade diária, então, por esse motivo, podemos criar uma coluna nova que nos auxilia a identificar as semanas devidamente. Cada semana vai de segunda a domingo e é identificada pela data da sua segunda-feira, primeiro dia útil, a coluna `semana`.

Também é interessante de ser criada a coluna `no_semana`, que numera as semanas em ordem e nos ajudará posteriormente na ordenação das semanas e na definição dos cortes para divisão dos dados.

Além disso, criaremos agregações a partir de diversas métricas nos dados atuais que nos auxiliarão na análise posterior.

In [441]:
# Agrupamento semanal (semana de segunda a domingo, identificada pela segunda-feira)
def inicio_da_semana(datas):
    """Devolve a segunda-feira da semana de cada data."""
    return (datas - pd.to_timedelta(datas.dt.weekday, unit='D')).dt.normalize()

def numero_semana(semanas, primeira_semana):
    """Retorna o número da semana contado a partir da primeira semana dos dados."""
    return ((semanas - primeira_semana).dt.days // 7 + 1).astype(int)

In [442]:
# Agrupamento de pedidos
df_pedidos['semana'] = inicio_da_semana(df_pedidos['data_pedido'])
PRIMEIRA_SEMANA = df_pedidos['semana'].min()

pedidos_semanal = (
    df_pedidos
    .groupby(['id_produto', 'semana'])
    .agg(
        qtd_total=('quantidade', 'sum'),
        qtd_media=('quantidade', 'mean'),
        qtd_mediana=('quantidade', 'median'),
        qtd_min=('quantidade', 'min'),
        qtd_max=('quantidade', 'max'),
        n_pedidos=('quantidade', 'count'),
        n_dias_com_pedido=('data_pedido', 'nunique'),
    )
    .reset_index()
)

pedidos_semanal.insert(2, 'no_semana', numero_semana(pedidos_semanal['semana'], PRIMEIRA_SEMANA))
pedidos_semanal = pedidos_semanal.sort_values(by=['semana', 'id_produto'])

In [443]:
# Garante que não tenha duplicamento ou perda de informação no agrupamento
assert pedidos_semanal['qtd_total'].sum() == df_pedidos['quantidade'].sum()
assert pedidos_semanal['n_pedidos'].sum() == len(df_pedidos)
assert not pedidos_semanal.duplicated(['id_produto', 'semana']).any()
assert pedidos_semanal.groupby('semana')['no_semana'].nunique().eq(1).all()

print(f"Formato: {pedidos_semanal.shape[0]} linhas ({pedidos_semanal['id_produto'].nunique()} produtos, {pedidos_semanal['semana'].nunique()} semanas)")
pedidos_semanal

Formato: 3617 linhas (340 produtos, 14 semanas)


,id_produto,semana,no_semana,qtd_total,qtd_media,qtd_mediana,qtd_min,qtd_max,n_pedidos,n_dias_com_pedido
16,1069819396768,2025-12-29,1,37.00,3.70,2.00,1.00,12.00,10,4
30,1113025112153,2025-12-29,1,9.00,3.00,4.00,1.00,4.00,3,1
52,1155835339892,2025-12-29,1,25.00,1.92,1.00,1.00,7.00,13,3
72,1176794648575,2025-12-29,1,60.00,30.00,30.00,30.00,30.00,2,2
77,1181121050608,2025-12-29,1,8.00,1.33,1.00,1.00,2.00,6,4
...,...,...,...,...,...,...,...,...,...,...
3570,9850160558162,2026-03-30,14,81.00,3.68,2.00,1.00,20.00,22,3
3580,9946963103349,2026-03-30,14,3.00,1.00,1.00,1.00,1.00,3,2
3590,9952616835809,2026-03-30,14,1.00,1.00,1.00,1.00,1.00,1,1
3602,9985808029395,2026-03-30,14,148.00,4.00,2.00,1.00,20.00,37,3


#### Dataset de produtos

In [444]:
print("====== INFORMAÇÕES GERAIS ======")
print(f"Formato: {df_produtos.shape[0]} linhas e {df_produtos.shape[1]} colunas")
print(f"Uso de memória: {df_produtos.memory_usage(deep=True).sum() / 1024**2:.2f} Mb")

====== INFORMAÇÕES GERAIS ======
Formato: 6453 linhas e 6 colunas
Uso de memória: 1.19 Mb


In [445]:
print("====== INFORMAÇÕES DETALHADAS ======")
print(df_produtos.info())

====== INFORMAÇÕES DETALHADAS ======
<class 'pandas.DataFrame'>
RangeIndex: 6453 entries, 0 to 6452
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_produto       6453 non-null   int64  
 1   curva            6453 non-null   str    
 2   categoria        6453 non-null   str    
 3   estoque          6453 non-null   int64  
 4   indice_de_preco  5628 non-null   float64
 5   data_coleta      6453 non-null   str    
dtypes: float64(1), int64(2), str(3)
memory usage: 302.6 KB
None


Assim como no dataset anterior, também podemos observar que `data_pedido` está sendo classificado como de tipo `object`, e pelo mesmo motivo da última modificação, também iremos convertê-lo para um tipo especificamente `datetime`.

Além disso, pela mesma razão que anteriormente, também aplicaremos a transformação de `id_produto` de um atributo numérico para uma string.

Aplicando a transformação:

In [446]:
df_produtos['data_coleta'] = pd.to_datetime(df_produtos['data_coleta'])
df_produtos['id_produto'] = df_produtos['id_produto'].astype(str)

In [447]:
print(df_produtos.info())

<class 'pandas.DataFrame'>
RangeIndex: 6453 entries, 0 to 6452
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_produto       6453 non-null   str           
 1   curva            6453 non-null   str           
 2   categoria        6453 non-null   str           
 3   estoque          6453 non-null   int64         
 4   indice_de_preco  5628 non-null   float64       
 5   data_coleta      6453 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 302.6 KB
None


Verificando novamente os tipos de dados:

In [448]:
print(df_produtos.info())

<class 'pandas.DataFrame'>
RangeIndex: 6453 entries, 0 to 6452
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_produto       6453 non-null   str           
 1   curva            6453 non-null   str           
 2   categoria        6453 non-null   str           
 3   estoque          6453 non-null   int64         
 4   indice_de_preco  5628 non-null   float64       
 5   data_coleta      6453 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 302.6 KB
None


Para os produtos, também aplicamos a mesma ideia de resumir os dias de cada semana em uma linha por produto e semana, usando as mesmas funções, permitindo que as tabelas possam ser unidas depois por (`id_produto`, `semana`).

Dessa vez, as agregações criadas foram seguindo o índice de preço na semana. Como mencionado pela equipe da Praso, esse índice de preço pode ser nulo quando não existem dados coletados para aquele período, então esses índices são "amortizado" pela agregação em casos em que apenas alguns dias da semana não possuem esse valor. Já quando todos os dias da semana são nulos, todas as agregações também serão.

Nessa etapa, ainda não realizamos nenhum tipo de técnica de tratamento desses valores nulos/ausentes, visto que o mecanismo e a forma de tratamento deve ser decidida na etapa de pré-processamento, quando estivermos usando apenas o conjunto de treino, buscando evitar vazamento de dados que possam comprometer as predições do modelo.

In [449]:
# Agrupamento de produtos
df_produtos['semana'] = inicio_da_semana(df_produtos['data_coleta'])

produtos_semanal = (
    df_produtos
    .groupby(['id_produto', 'semana'])
    .agg(
        curva=('curva', 'last'),
        categoria=('categoria', 'last'),
        ip_mediano=('indice_de_preco', 'median'),
        ip_medio=('indice_de_preco', 'mean'),
        ip_min=('indice_de_preco', 'min'),
        ip_max=('indice_de_preco', 'max'),
        n_dias_coleta=('data_coleta', 'nunique'),
        ips_nao_nulos=('indice_de_preco', 'count'),  # dias com índice não nulo
        pct_dias_sem_estoque=('estoque', lambda s: 1 - s.mean()),
    )
    .reset_index()
)

produtos_semanal.insert(2, 'no_semana', numero_semana(produtos_semanal['semana'], PRIMEIRA_SEMANA))
produtos_semanal = produtos_semanal.sort_values(by=['semana', 'id_produto'])

In [450]:
# Garante que não tenha duplicamento ou perda de informação no agrupamento
assert produtos_semanal['n_dias_coleta'].sum() == len(df_produtos)
assert not produtos_semanal.duplicated(['id_produto', 'semana']).any()
assert produtos_semanal.groupby('semana')['no_semana'].nunique().eq(1).all()
assert (produtos_semanal['no_semana'] >= 1).all()

print(f"Formato: {produtos_semanal.shape[0]} linhas ({produtos_semanal['id_produto'].nunique()} produtos, {produtos_semanal['semana'].nunique()} semanas)")
produtos_semanal

Formato: 1051 linhas (103 produtos, 14 semanas)


,id_produto,semana,no_semana,curva,categoria,ip_mediano,ip_medio,ip_min,ip_max,n_dias_coleta,ips_nao_nulos,pct_dias_sem_estoque
0,1010954545613,2025-12-29,1,A,Categoria 27,3.19,3.19,3.19,3.19,5,5,0.00
14,1113025112153,2025-12-29,1,B,Categoria 39,0.87,0.87,0.87,0.87,5,5,0.00
33,1176794648575,2025-12-29,1,B,Categoria 1,1.13,1.13,1.13,1.13,2,2,0.00
37,1181121050608,2025-12-29,1,B,Categoria 7,1.10,1.10,1.10,1.10,5,5,0.00
51,1271489837738,2025-12-29,1,B,Categoria 32,NaN,NaN,NaN,NaN,5,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
998,9510925091410,2026-03-30,14,B,Categoria 9,1.10,1.10,1.10,1.10,3,3,0.00
1018,9613953741219,2026-03-30,14,B,Categoria 4,0.98,1.01,0.98,1.06,3,3,0.00
1033,9725294492933,2026-03-30,14,B,Categoria 13,1.16,1.16,1.16,1.16,3,3,0.00
1045,9765282948827,2026-03-30,14,B,Categoria 35,1.29,1.29,1.29,1.29,3,3,0.00


#### Junção dos conjuntos de dados

In [451]:
df_pedidos_semana = pd.merge(
    pedidos_semanal,
    produtos_semanal,
    on=['no_semana', 'semana', 'id_produto'],
    how='left',
    validate='1:1'
)

df_pedidos_semana

,id_produto,semana,no_semana,qtd_total,qtd_media,qtd_mediana,qtd_min,qtd_max,n_pedidos,n_dias_com_pedido,curva,categoria,ip_mediano,ip_medio,ip_min,ip_max,n_dias_coleta,ips_nao_nulos,pct_dias_sem_estoque
0,1069819396768,2025-12-29,1,37.00,3.70,2.00,1.00,12.00,10,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1113025112153,2025-12-29,1,9.00,3.00,4.00,1.00,4.00,3,1,B,Categoria 39,0.87,0.87,0.87,0.87,5.00,5.00,0.00
2,1155835339892,2025-12-29,1,25.00,1.92,1.00,1.00,7.00,13,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1176794648575,2025-12-29,1,60.00,30.00,30.00,30.00,30.00,2,2,B,Categoria 1,1.13,1.13,1.13,1.13,2.00,2.00,0.00
4,1181121050608,2025-12-29,1,8.00,1.33,1.00,1.00,2.00,6,4,B,Categoria 7,1.10,1.10,1.10,1.10,5.00,5.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612,9850160558162,2026-03-30,14,81.00,3.68,2.00,1.00,20.00,22,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3613,9946963103349,2026-03-30,14,3.00,1.00,1.00,1.00,1.00,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3614,9952616835809,2026-03-30,14,1.00,1.00,1.00,1.00,1.00,1,1,B,Categoria 26,NaN,NaN,NaN,NaN,3.00,0.00,0.33
3615,9985808029395,2026-03-30,14,148.00,4.00,2.00,1.00,20.00,37,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ➗ | Divisão em conjunto de treinamento, validação e teste

Considerando que estamos trabalhando com dados com uma temporalidade relevante, é necessário que realizemos o agrupamento dos dados considerando esse aspecto. Portanto, a divisão entre treino, validação e teste deve respeitar rigorosamente a ordem cronológica dos dados.

Além disso, realizaremos esse processo antes da análise exploratória e do pré-processamento para garantir que não irá ocorrer vazamento dos dados nessas etapas, ou seja, qualquer estatística ou transformação dos dados será baseada apenas nos dados de treino.

Antes dividir a base de dados final, precisamos verificar se todas as semanas são comparáveis entre si. Como os dados vão de 31/12/2025 (quarta-feira) a 01/04/2026 (quarta-feira), e cada semana vai de segunda a domingo, é esperado que as semanas das extremidades (primeira e última) não têm todos os dias. Verificando:

In [452]:
resumo_semanas = (
    df_pedidos
    .groupby('semana')
    .agg(
        primeiro_dia=('data_pedido', 'min'),
        ultimo_dia=('data_pedido', 'max'),
        dias_com_dados=('data_pedido', 'nunique'),
        qtd_total=('quantidade', 'sum'),
        n_pedidos=('quantidade', 'count'),
        n_produtos=('id_produto', 'nunique'),
    )
    .reset_index()
)

resumo_semanas.insert(1, 'no_semana', numero_semana(resumo_semanas['semana'], PRIMEIRA_SEMANA))
resumo_semanas['qtd_por_dia'] = resumo_semanas['qtd_total'] / resumo_semanas['dias_com_dados']
resumo_semanas['semana_parcial'] = resumo_semanas['dias_com_dados'] < 7

resumo_semanas

,semana,no_semana,primeiro_dia,ultimo_dia,dias_com_dados,qtd_total,n_pedidos,n_produtos,qtd_por_dia,semana_parcial
0,2025-12-29,1,2025-12-31,2026-01-04,5,9198.00,967,202,1839.60,True
1,2026-01-05,2,2026-01-05,2026-01-11,7,38819.00,3911,266,5545.57,False
2,2026-01-12,3,2026-01-12,2026-01-18,7,34025.00,3227,275,4860.71,False
3,2026-01-19,4,2026-01-19,2026-01-25,7,36290.00,3540,264,5184.29,False
4,2026-01-26,5,2026-01-26,2026-02-01,7,35890.00,3468,258,5127.14,False
5,2026-02-02,6,2026-02-02,2026-02-08,7,40435.00,3736,269,5776.43,False
6,2026-02-09,7,2026-02-09,2026-02-15,7,33465.00,3356,271,4780.71,False
7,2026-02-16,8,2026-02-16,2026-02-22,7,27530.00,2908,250,3932.86,False
8,2026-02-23,9,2026-02-23,2026-03-01,7,36680.00,3681,266,5240.00,False
9,2026-03-02,10,2026-03-02,2026-03-08,7,34936.00,3562,262,4990.86,False


Conforme levantada a hipótese anterior, a semana 1 (31/12 a 04/01) possui apenas 5 dias e a semana 14 (30/03 a 01/04) possui apenas 3, enquanto todas as outras possuem os 7 dias. Por isso, o `qtd_total` dessas duas semanas é bem menor que o das semanas completas devido à observação de menos dias e não por uma baixa natural, o que pode enviezar a etapa de modelagem.

Uma alternativa seria dividir o total pelo número de dias (`qtd_por_dia`), mas isso só funcionaria se todos os dias da semana vendessem uma quantidade parecida. Também podemos verificar isso:

In [453]:
semanas_completas = resumo_semanas.loc[~resumo_semanas['semana_parcial'], 'semana']
pedidos_completos = df_pedidos[df_pedidos['semana'].isin(semanas_completas)]

pct_por_dia = (
    pedidos_completos.groupby(pedidos_completos['data_pedido'].dt.dayofweek)['quantidade'].sum()
    / pedidos_completos['quantidade'].sum() * 100
)
pct_por_dia.index = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
pct_por_dia.to_frame('% do volume semanal')

,% do volume semanal
Seg,24.88
Ter,18.88
Qua,18.94
Qui,16.30
Sex,13.35
Sáb,3.53
Dom,4.12


Dessa forma, optamos por desconsiderar as duas semanas incompletas (1 e 14) da base de modelagem por cobrirem partes diferentes da semana que não apresentam uma amostra representativa. Como visto acima, a semana 1 inclui só quarta a domingo (~56% do volume semanal) e ainda contém a virada de ano (31/12 e 01/01), que tem pouco volume. Já a semana 14 inclui só segunda a quarta (~63% do volume semanal).

In [454]:
# Mantém apenas as semanas completas (descarta a primeira e a última)
nos_semanas_completas = resumo_semanas.loc[~resumo_semanas['semana_parcial'], 'no_semana']
dados = df_pedidos_semana[df_pedidos_semana['no_semana'].isin(nos_semanas_completas)].copy()

print(f"Formato: {dados.shape[0]} linhas ({dados['id_produto'].nunique()} produtos, semanas {dados['no_semana'].min()} a {dados['no_semana'].max()})")
dados

Formato: 3168 linhas (340 produtos, semanas 2 a 13)


,id_produto,semana,no_semana,qtd_total,qtd_media,qtd_mediana,qtd_min,qtd_max,n_pedidos,n_dias_com_pedido,curva,categoria,ip_mediano,ip_medio,ip_min,ip_max,n_dias_coleta,ips_nao_nulos,pct_dias_sem_estoque
202,1034629075323,2026-01-05,2,1.00,1.00,1.00,1.00,1.00,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
203,1040193335281,2026-01-05,2,1.00,1.00,1.00,1.00,1.00,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,1069819396768,2026-01-05,2,377.00,4.83,3.00,1.00,36.00,78,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,1113025112153,2026-01-05,2,17.00,2.83,2.50,1.00,6.00,6,3,B,Categoria 39,0.87,0.87,0.87,0.87,7.00,7.00,0.00
206,1148945761075,2026-01-05,2,1.00,1.00,1.00,1.00,1.00,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3365,9850160558162,2026-03-23,13,118.00,2.88,1.00,1.00,20.00,41,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3366,9946963103349,2026-03-23,13,9.00,1.50,1.50,1.00,2.00,6,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3367,9952616835809,2026-03-23,13,4.00,1.00,1.00,1.00,1.00,4,4,B,Categoria 26,NaN,NaN,NaN,NaN,7.00,0.00,0.00
3368,9985808029395,2026-03-23,13,174.00,4.46,3.00,1.00,15.00,39,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Com as semanas completas, podemos realizar os dados pelo número da semana, sempre respeitando a cronologia para evitar que avaliemos dados do passado com treinamento futuro.

- Treino: semanas 2 a 7 (6 semanas ~ 50% dos registros)
- Validação: semanas 8 a 10 (3 semanas ~ 25% dos registros)
- Teste: semanas 11 a 13 (3 semanas ~ 25% dos registros)

In [455]:
df_treino = dados[dados['no_semana'] <= 7]
df_validacao = dados[dados['no_semana'].between(8, 10)]
df_teste = dados[dados['no_semana'] >= 11]

assert len(df_treino) + len(df_validacao) + len(df_teste) == len(dados) # Sem perda de dados

for nome, df in [('Treino', df_treino), ('Validação', df_validacao), ('Teste', df_teste)]:
    print(f"{nome}: semanas {df['no_semana'].min()} a {df['no_semana'].max()} ({len(df)} registros)")

Treino: semanas 2 a 7 (1603 registros)
Validação: semanas 8 a 10 (778 registros)
Teste: semanas 11 a 13 (787 registros)


Vamos exportar os dados para serem usados em outros notebooks.

Ainda não exportaremos o conjunto de treinamento, dado que ele ainda será explorado e pré-processado.

In [456]:
df_validacao.to_csv('../data/treino.csv', index=False)
df_teste.to_csv('../data/teste.csv', index=False)

### Análise Exploratória Estrutural

Inicialmente, podemos realizar uma análise inicial do formato dos dados e dos tipos das *features* que serão trabalhadas para que seja possível realizar uma análise posterior de aspectos mais específicos e intrínssecos a cada *feature*.